In [0]:
import pandas as pd
import time
from pyspark.sql import functions as f
from pyspark.sql import types as t
import random
from src.utils.validations import *
from src.utils.file import *
from datetime import datetime
from pyspark.sql import SparkSession
import os
from zoneinfo import ZoneInfo


In [0]:
def read_csv(path: str, use_spark=False, **kwargs):

        try:

            if use_spark:

                df = (
                    spark.read
                    .options(**kwargs)
                    .csv(path)
                    .withColumn(
                        "source_file",
                        f.element_at(
                            f.split(
                                f.col("_metadata.file_path"),
                                "/"
                            ),
                            -1
                        )
                    )
                )

                # força validação
                df.limit(1).collect()

            else:

                df = pd.read_csv(path, **kwargs)

                df["source_file"] = os.path.basename(path)

            return {
                "status": True,
                "df": df,
                "erro": None
            }

        except Exception as e:

            return {
                "status": False,
                "df": None,
                "erro": str(e)
            }

In [0]:
dir_path = r"/Volumes/workspace/callcenter/disc_volumes/landing/"

--- VALIDA EXISTENCIA DE ARQUIVO DO DIA ATUAL

In [0]:

dir_path = r"/Volumes/workspace/callcenter/disc_volumes/landing/"
df_processed_file = spark.table(r"workspace.quality.execucao_arquivo")

df_processed_file.display()


files = dbutils.fs.ls(dir_path)




for file in files:
    print(file.name)



data_hoje = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).strftime("%Y%m%d")


data_hoje = datetime.now().strftime("%Y%m%d")

path_hoje = f"{dir_path}discagem_{data_hoje}_*.csv"

df_carga = read_csv(path=path_hoje, header=True, use_spark=True, sep=";")

In [0]:
df_carga.display()